**ENTRENAMIENTO DEL DETECTOR DE MATRÍCULAS CON YOLO**

En este cuaderno se describe cómo se entrenó el detector de matrículas (`best-yolo.pt`) que se utiliza después en el cuaderno de lectura y seguimiento en vídeo.

El flujo de trabajo se estructura en:
1. **Comprobación de GPU**: Verificación de que PyTorch dispone de CUDA para acelerar el entrenamiento.
2. **Entrenamiento**: *Fine-tuning* de `yolo11s` (preentrenado en COCO) sobre el conjunto de matrículas, con una única clase (`plate`).
3. **Evaluación**: Métricas de precisión, recall y mAP sobre el conjunto de validación, junto con las curvas de entrenamiento y la matriz de confusión.
4. **Prueba de inferencia**: Predicción visual sobre una imagen para comprobar el resultado.

**CONFIGURACIÓN INICIAL: COMPROBACIÓN DE GPU**

Antes de entrenar se comprueba si PyTorch detecta una GPU con CUDA. El entrenamiento a 960 px de resolución es muy lento en CPU, por lo que conviene confirmar el dispositivo disponible.

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

**ENTRENAMIENTO DEL MODELO**

Se parte de `yolo11s.pt`, preentrenado en COCO, y se ajusta (*fine-tuning*) sobre el dataset de matrículas definido en `datos/data.yaml`. Los hiperparámetros principales son:
- **`imgsz=960`**: Resolución alta, porque las matrículas ocupan una zona muy pequeña de la imagen.
- **`batch=8`**: Tamaño de lote ajustado a la memoria de la GPU (se reduce si aparece *CUDA out of memory*).
- **`patience=20`**: Parada temprana si la métrica de validación no mejora durante 20 épocas.
- **`fliplr=0.0`**: Se desactiva el espejado horizontal, ya que una matrícula espejada no es un caso real.
- **`degrees=5`, `hsv_v=0.4`**: Aumentado de datos con ligeras rotaciones y variaciones de brillo.
- **`close_mosaic=10`**: Se desactiva el aumentado *mosaic* en las últimas 10 épocas para estabilizar el ajuste final.

In [ ]:
import torch
from pathlib import Path
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

results = model.train(
    data="datos/data.yaml",    # dataset: rutas de train/val y nombres de clase
    epochs=100,                # numero maximo de epocas
    imgsz=960,                 # matriculas pequeñas; si falta memoria, baja a 800 o 640
    batch=8,                   # si sale "CUDA out of memory", baja a 4
    device=0 if torch.cuda.is_available() else "cpu",
    patience=20,               # parada temprana tras 20 epocas sin mejora
    cache=True,                # cachea las imagenes en RAM; si te quedas sin RAM, pon False
    fliplr=0.0,                # no espejar matriculas
    degrees=5,                 # rotacion aleatoria maxima de +-5 grados
    hsv_v=0.4,                 # variacion de brillo (canal V del espacio HSV)
    close_mosaic=10,           # desactiva mosaic en las ultimas 10 epocas
    name="yolo_plate",         # nombre de la carpeta de resultados
)

RUTA = Path(model.trainer.save_dir)
print("Resultados en:", RUTA)

**EVALUACIÓN DEL MODELO ENTRENADO**

Nos quedamos con la mejor versión `best.pt` y se evalúa sobre el conjunto de validación. Se obtienen:
- **Precision**: Proporción de detecciones que son realmente matrículas.
- **Recall**: Proporción de matrículas reales que el modelo consigue detectar.
- **mAP50 / mAP50-95**: Precisión media con IoU de 0.5 y promediada entre 0.5 y 0.95, respectivamente.

Además se muestran las curvas de entrenamiento (`results.png`) y la matriz de confusión (`confusion_matrix.png`).

In [ ]:
from IPython.display import Image, display

# Carga los mejores pesos guardados durante el entrenamiento
best = YOLO(RUTA / "weights" / "best.pt")

# Evaluacion sobre el conjunto de validacion definido en data.yaml
metricas = best.val(data="datos/data.yaml")
print(f"Precision: {metricas.box.mp:.3f} | Recall: {metricas.box.mr:.3f} | "
      f"mAP50: {metricas.box.map50:.3f} | mAP50-95: {metricas.box.map:.3f}")

# Curvas de perdida y metricas por epoca
display(Image(filename=str(RUTA / "results.png")))
# Matriz de confusión del detector
display(Image(filename=str(RUTA / "confusion_matrix.png")))

**PRUEBA DE INFERENCIA**

Como comprobación final se ejecuta el detector sobre una imagen del dataset, con un umbral de confianza de 0.25, y se dibujan las cajas predichas junto con el número de matrículas detectadas.

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Primera imagen de la carpeta de imagenes del dataset
img_path = "datos/images/" + next(iter(sorted(p.name for p in (Path("datos") / "images").iterdir())))

# Inferencia con el mejor modelo (conf mínima 0.25, misma resolucion que en el entrenamiento)
r = best.predict(img_path, conf=0.25, imgsz=960)[0]

# r.plot() devuelve la imagen con las cajas en BGR; se convierte a RGB para matplotlib
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
print("Placas detectadas:", len(r.boxes))